# Demo 12. Gaussian Elimination and LU with Partial Pivoting

**Standard 5** (direct methods), Week 4.

Gaussian elimination solves $Ax=b$. This demo presents it as an LU factorization, shows how a small pivot degrades the unpivoted method, how partial pivoting corrects it, and confirms the $\tfrac{2}{3}n^3$ operation count.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider, IntSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)
np.set_printoptions(precision=4, suppress=True)

## Pivoting

Elimination replaces row $i$ by $\mathrm{row}_i - m_{ik}\,\mathrm{row}_k$ with multiplier $m_{ik} = a_{ik}/a_{kk}$. A small pivot $a_{kk}$ makes $m_{ik}$ large and amplifies rounding error. Partial pivoting moves the largest available entry into the pivot position, giving $|m_{ik}| \le 1$. The result is $PA = LU$ with $P$ a permutation, $L$ unit lower triangular, and $U$ upper triangular, after which $Ax=b$ reduces to two triangular solves.

In [ ]:
# ---------------------------------------------------------------------------
# Gaussian elimination without pivoting (no pivoting)
# ---------------------------------------------------------------------------
# Eliminate column by column: use row k to zero out the entries below the pivot
# A[k,k]. The danger: if a pivot is tiny, dividing by it amplifies rounding error.

def lu_no_pivot(A):
    """LU factorisation with NO pivoting: A = L U (may be unstable / may fail).

    Returns lower-triangular L (unit diagonal) and upper-triangular U.
    Breaks down if any pivot is exactly zero, and loses accuracy if one is tiny.
    """
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    for k in range(n):                       # eliminate below pivot U[k,k]
        for i in range(k + 1, n):
            m = U[i, k] / U[k, k]            # multiplier (danger if U[k,k] tiny)
            L[i, k] = m
            U[i, k:] -= m * U[k, k:]
    return L, U

In [ ]:
# ---------------------------------------------------------------------------
# Gaussian elimination with partial pivoting
# ---------------------------------------------------------------------------
# Before eliminating column k, swap in the row whose entry in that column has the
# largest absolute value. This keeps every multiplier <= 1 in magnitude, which
# controls error growth. Result: P A = L U, where P records the row swaps.

def lu_partial_pivot(A):
    """LU factorisation with partial pivoting:  P A = L U.

    Returns P (permutation matrix), L (unit lower triangular), U (upper triangular).
    Partial pivoting makes this the standard, backward-stable direct solver.
    """
    A = np.array(A, dtype=float)
    n = A.shape[0]
    U = A.copy()
    L = np.eye(n)
    P = np.eye(n)
    for k in range(n):
        # --- choose the pivot: largest |entry| in column k, rows k..n-1 ---
        p = k + np.argmax(np.abs(U[k:, k]))
        if p != k:                           # swap rows k and p everywhere
            U[[k, p], :] = U[[p, k], :]
            P[[k, p], :] = P[[p, k], :]
            if k > 0:                        # also swap the already-built part of L
                L[[k, p], :k] = L[[p, k], :k]
        # --- eliminate below the (now largest) pivot ---
        for i in range(k + 1, n):
            m = U[i, k] / U[k, k]
            L[i, k] = m
            U[i, k:] -= m * U[k, k:]
    return P, L, U

def lu_solve(P, L, U, b):
    """Solve A x = b given P A = L U, by forward then back substitution."""
    b = np.asarray(b, float)
    pb = P @ b
    n = len(b)
    # forward substitution: L y = P b
    y = np.zeros(n)
    for i in range(n):
        y[i] = pb[i] - L[i, :i] @ y[:i]
    # back substitution: U x = y
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x

def flop_count(n):
    """Leading-order operation count for LU of an n x n matrix: ~ 2/3 n^3."""
    return (2.0 / 3.0) * n**3

In [ ]:
# ---------------------------------------------------------------------------
# Why pivoting matters: a tiny pivot wrecks the naive method
# ---------------------------------------------------------------------------
# The classic 2x2:   [ eps  1 ] [x1]   [1]
#                    [  1   1 ] [x2] = [2]
# For small eps the true answer is close to x = (1, 1). Naive elimination divides
# by eps and loses digits; partial pivoting swaps the rows first and stays sharp.

def _solve_LU_plain(L, U, b):
    """Forward/back substitution for an un-permuted L, U (no pivoting)."""
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]         # forward: L y = b
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]   # back: U x = y
    return x

def show_pivoting(log10_eps=-14.0):
    """Solve the tiny-pivot 2x2 both ways and compare each to the exact solution.

    Exact solution (from Cramer's rule, det = eps - 1):
        x1 = (1 - 2*eps)/(1 - eps),   x2 = (2*eps - 1)/(eps - 1).
    """
    eps = 10.0 ** log10_eps
    A = np.array([[eps, 1.0], [1.0, 1.0]])
    b = np.array([1.0, 2.0])

    x1 = (1 - 2 * eps) / (1 - eps)
    x2 = (2 * eps - 1) / (eps - 1)
    x_true = np.array([x1, x2])

    L, U = lu_no_pivot(A)                       # naive factorisation
    x_naive = _solve_LU_plain(L, U, b)

    P, Lp, Up = lu_partial_pivot(A)             # pivoted factorisation
    x_piv = lu_solve(P, Lp, Up, b)

    print(f"eps = {eps:.1e}")
    print(f"  exact             x = {x_true}")
    print(f"  naive (no pivot)  x = {x_naive}   error = {np.linalg.norm(x_naive - x_true):.2e}")
    print(f"  partial pivoting  x = {x_piv}   error = {np.linalg.norm(x_piv - x_true):.2e}")

show_pivoting(-14.0)

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: shrink eps and watch the naive error grow while pivoting holds
# ---------------------------------------------------------------------------
interact(
    show_pivoting,
    log10_eps=FloatLogSlider(value=1e-14, base=10, min=-16, max=-1, step=1,
                             description="eps", readout_format=".0e"),
);

In [ ]:
# ---------------------------------------------------------------------------
# Operation count: LU costs about (2/3) n^3 flops
# ---------------------------------------------------------------------------
def show_flops(n=50):
    """Print the leading-order flop count and verify our LU matches numpy."""
    rng = np.random.default_rng(1)
    A = rng.standard_normal((n, n))
    b = rng.standard_normal(n)
    P, L, U = lu_partial_pivot(A)
    x = lu_solve(P, L, U, b)
    print(f"n = {n}:  ~(2/3)n^3 = {flop_count(n):.3e} flops")
    print(f"  || P A - L U ||  = {np.linalg.norm(P @ A - L @ U):.2e}")
    print(f"  || A x - b ||    = {np.linalg.norm(A @ x - b):.2e}")

interact(show_flops, n=IntSlider(min=2, max=120, step=1, value=50, description="n"));

## Summary

- LU factorization records Gaussian elimination as $A = LU$, or $PA=LU$ with pivoting.
- A small pivot degrades the unpivoted method; partial pivoting bounds all multipliers by 1 and restores accuracy. The error gap widens as $\varepsilon$ decreases.
- The cost is $\tfrac{2}{3}n^3$ operations; each additional right-hand side costs $O(n^2)$.
- The residual $\lVert Ax-b\rVert$ and $\lVert PA-LU\rVert$ remain small.